# Import Files

In [43]:
# pip install neurokit2
import json
import numpy as np
import matplotlib.pyplot as plt
import neurokit2 as nk # Source https://neuropsychology.github.io/NeuroKit/functions/ecg.html
import pandas as pd
import matplotlib.pyplot as plt

##### Code to Import Other Files from the Root Folder


In [44]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[0]
sys.path.append(str(project_root))
# Now you can import modules from the project root

##### Import the Feature Engineering Algorithm


In [45]:
from feature_engineering.feature_engineering import extract_features_from_segment


### Defining the Possible Files that we can work with
We will iteratively go through these one by one and add the outputs to the pandas dataframe which contains the results and thereafter use that dataframe to create the input for the Random Forest algorithm

In [46]:
data_file_array=["tache_lacasse_1min.txt",          # <- normal data
                 "CFA4_1min_standard_position.json", # <- normal data
                "CFA2_1min_standard_position.json", # <- abnormal data
                "CFA12_1min_standard_position.json", # <- borderline data
                ]


In [47]:
class ECGProcessor:
    def __init__(self, data_file):
        self.data_file = data_file
        self.ecg_json = None
        self.ecg_data = None
        self.fs = None
        self.num_leads = None
        self.ecg_mv = None
        self.lead_I = None
        self.lead_II = None
        self.lead_III = None
        self.lead_aVR = None
        self.lead_aVL = None
        self.lead_aVF = None
        self.lead_V1 = None
        self.lead_V2 = None
        self.lead_V3 = None
        self.lead_V4 = None
        self.lead_V5 = None
        self.lead_V6 = None
        self.leads = {}
        self.ecg_df = None


# === Function to extract data based on the header info ===
    def extract_ecg_data(self):

        # === Load JSON ECG data ===
        with open(self.data_file, "r") as f:
            self.ecg_json = json.load(f)

        # Here we’re picking out the key facts from the JSON file's structure:
        # Some of this data includes:
        # ecg_data = all the raw electrical readings from the electrodes (the heartbeat signals).
        # fs = how many times per second the signal was measured (the sampling rate, here 500 times per second).
        # scale = how to convert the raw numbers into meaningful electrical units (millivolts).
        # num_leads = how many electrodes were used (8 in this case).

        self.ecg_data = np.array(self.ecg_json["ecg"])
        self.fs = self.ecg_json["header"]["samplingRate"]          # Sampling frequency (Hz)
        self.scale = self.ecg_json["header"]["unitsPerMv"]         # Conversion factor (counts per mV)
        self.num_leads = self.ecg_json["footer"]["numberOfLeads"]  # In our case, this is 8
        self.duration_ms = self.ecg_json["footer"]["duration"]     # Duration in ms
        

    def convert_to_mV(self):
        # The machine originally saves numbers that represent electrical voltage, but not in “real world” units.
        # Dividing by the "scale" value turns them into millivolts (mV), which is the standard medical unit for ECG amplitude.
        # === Convert raw data to mV ===
        self.ecg_mv = self.ecg_data / self.scale

    def create_12_lead_ecg(self):
    # Original 8 leads (by index)
        vec0 = self.ecg_mv[:, 0]  # Lead I
        vec1 = self.ecg_mv[:, 1]  # Lead II
        vec2 = self.ecg_mv[:, 2]  # V1
        vec3 = self.ecg_mv[:, 3]  # V2
        vec4 = self.ecg_mv[:, 4]  # V3
        vec5 = self.ecg_mv[:, 5]  # V4
        vec6 = self.ecg_mv[:, 6]  # V5
        vec7 = self.ecg_mv[:, 7]  # V6

    # Limb leads
        self.lead_I   = vec0
        self.lead_II  = vec1
        self.lead_III = vec1 - vec0

        # Augmented limb leads
        self.lead_aVR = -(vec0 + vec1) / 2
        self.lead_aVL = vec0 - vec1 / 2
        self.lead_aVF = vec1 - vec0 / 2

        # Precordial leads
        self.lead_V1 = vec2
        self.lead_V2 = vec3
        self.lead_V3 = vec4
        self.lead_V4 = vec5
        self.lead_V5 = vec6
        self.lead_V6 = vec7

        # 12-lead ECG dictionary
        ecg_12lead = {
            "I":   self.lead_I,
            "II":  self.lead_II,
            "III": self.lead_III,
            "aVR": self.lead_aVR,
            "aVL": self.lead_aVL,
            "aVF": self.lead_aVF,
            "V1":  self.lead_V1,
            "V2":  self.lead_V2,
            "V3":  self.lead_V3,
            "V4":  self.lead_V4,
            "V5":  self.lead_V5,
            "V6":  self.lead_V6,
        }

        # Store this data into a giant DataFrame
        self.ecg_df = pd.DataFrame(ecg_12lead)
        t = np.arange(len(self.ecg_mv)) / self.fs
        self.ecg_df["time"] = t
        self.leads = ecg_12lead 



    # Process each lead independently, but keep everything in a dictionary.
    def process_ecg_lead(self,ecg_signal, fs):
        """
        Run NeuroKit ECG processing on a single lead.
        Returns signals DataFrame and info dict.
        """
        signals, info = nk.ecg_process(ecg_signal, sampling_rate=self.fs)
        return signals, info


    # Build labeled signal DataFrame (P, Q, R, S, T)
    def build_labeled_signal(self,signals):
        labels = np.full(len(signals), '', dtype=object)

        peak_columns = {
            'P': 'ECG_P_Peaks',
            'Q': 'ECG_Q_Peaks',
            'R': 'ECG_R_Peaks',
            'S': 'ECG_S_Peaks',
            'T': 'ECG_T_Peaks'
        }

        for label, col in peak_columns.items():
            if col in signals:
                labels[signals[col] == 1] = label

        df = pd.DataFrame({
            'ECG_Raw': signals['ECG_Raw'],
            'ECG_Clean': signals['ECG_Clean'],
            'Label': labels
        })

        return df

    # Segment beats individually using P-peaks
    def segment_by_r_peaks(self,labeled_df):
        p_indices = labeled_df.index[labeled_df['Label'] == 'P'].tolist()

        segments = []
        for i in range(len(p_indices) - 1):
            seg = labeled_df.loc[p_indices[i]:p_indices[i+1]].copy()
            segments.append(seg)

        return segments




    def extract_features(self):
        """
        Extract beat-level features from all 12 leads.
        Returns a DataFrame of all features.
        """

        all_features = []

        for lead_name, ecg_signal in self.leads.items():

            print(f"Processing lead {lead_name}")

            # 1. Process ECG
            signals, info = self.process_ecg_lead(ecg_signal, self.fs)

            # 2. Label peaks
            labeled_df = self.build_labeled_signal(signals)

            # 3. Segment beats
            segments = self.segment_by_r_peaks(labeled_df)

            # 4. Extract features per beat
            for beat_idx, segment in enumerate(segments):

                feat = extract_features_from_segment(segment, self.fs)

                # ---- Metadata ----
                feat["Lead"] = lead_name
                feat["Beat_Index"] = beat_idx
                feat["File_Name"] = self.data_file
                feat["Num_Samples"] = len(segment)

                all_features.append(feat)

        return pd.DataFrame(all_features)




In [48]:
import pandas as pd
from pathlib import Path
from scipy.signal import butter, filtfilt

# -------------------------------
# Step 0: Optional filtering function
# -------------------------------
def bandpass_filter(signal, fs, lowcut=0.5, highcut=40, order=4):
    """
    Apply a Butterworth bandpass filter to a 1D ECG signal.
    """
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, signal)

# -------------------------------
# Step 1: Find all JSON files
# -------------------------------
# Go up 2 levels from current working directory
project_root = Path.cwd().parents[1]

# Path to raw data
raw_data_path = project_root / "data" / "raw"

# Recursively find all JSON files in subdirectories
data_files = list(raw_data_path.rglob("*.json"))

print(f"Found {len(data_files)} JSON files to process.")

# -------------------------------
# Step 2: Initialize empty DataFrame
# -------------------------------
features_df = pd.DataFrame()

# -------------------------------
# Step 3: Iterate through all files
# -------------------------------
for data_file in data_files:
    print(f"Processing {data_file}...")
    
    # Initialize processor
    processor = ECGProcessor(str(data_file))
    
    # Extract ECG
    processor.extract_ecg_data()
    processor.convert_to_mV()
    processor.create_12_lead_ecg()
    
    # # -------------------------------
    # # Optional: Filter leads outside class
    # # -------------------------------
    # for lead_name in processor.leads:
    #     processor.leads[lead_name] = bandpass_filter(
    #         processor.leads[lead_name], fs=processor.fs
    #     )
    
    # Extract features
    features = processor.extract_features()
    
    # Concatenate into final DataFrame
    features_df = pd.concat([features_df, features], ignore_index=True)

print("Feature extraction complete.")
print(f"Total features extracted: {len(features_df)}")


Found 3 JSON files to process.
Processing /Users/fahim/Documents/GitHub/EKG Processing/data/raw/abnormal/CFA2_1min_standard_position.json...
Processing lead I
Processing lead II
Processing lead III
Processing lead aVR
Processing lead aVL
Processing lead aVF
Processing lead V1
Processing lead V2
Processing lead V3
Processing lead V4
Processing lead V5
Processing lead V6
Processing /Users/fahim/Documents/GitHub/EKG Processing/data/raw/normal/CFA4_1min_standard_position.json...
Processing lead I
Processing lead II
Processing lead III
Processing lead aVR
Processing lead aVL
Processing lead aVF
Processing lead V1
Processing lead V2
Processing lead V3
Processing lead V4
Processing lead V5
Processing lead V6
Processing /Users/fahim/Documents/GitHub/EKG Processing/data/raw/borderline/CFA12_1min_standard_position.json...
Processing lead I
Processing lead II
Processing lead III
Processing lead aVR
Processing lead aVL
Processing lead aVF
Processing lead V1
Processing lead V2
Processing lead V3
Pr

In [49]:
# features_df=pd.DataFrame()
# for data_file in data_file_array:
#     processor = ECGProcessor(data_file)
#     processor.extract_ecg_data()
#     processor.convert_to_mV()
#     processor.create_12_lead_ecg()
#     features = processor.extract_features()
#     features_df = pd.concat([features_df, features], ignore_index=True)

In [50]:
# processor = ECGProcessor("tache_lacasse_1min.txt")
# # processor = ECGProcessor("file.json")

# processor.extract_ecg_data()
# processor.convert_to_mV()
# processor.create_12_lead_ecg()

# print(processor.leads.keys())   # Should show 12 leads

# features_df = processor.extract_features()


In [51]:
processor.ecg_mv



array([[ 1.27439024, -0.56402439, -0.55182927, ..., -0.54878049,
         0.0945122 , -0.60060976],
       [ 1.2804878 , -0.56097561, -0.54573171, ..., -0.55182927,
         0.09756098, -0.6097561 ],
       [ 1.2804878 , -0.56097561, -0.53353659, ..., -0.53658537,
         0.10060976, -0.59146341],
       ...,
       [-0.06402439, -0.63719512,  2.42987805, ...,  1.29573171,
         1.20731707,  0.69512195],
       [-0.06707317, -0.63719512,  2.42073171, ...,  1.28963415,
         1.20731707,  0.69512195],
       [-0.06707317, -0.63414634,  2.41463415, ...,  1.28353659,
         1.20121951,  0.68902439]])

In [52]:
features_df


,T_estimated,P_Q,Q_R,R_S,S_T,P_R,QR_slope,RS_slope,AUC_full,AUC_QRS,Lead,Beat_Index,File_Name,Num_Samples
0,0,0.090,0.032,0.062,0.028,0.122,3.985376,-1.773498,0.011439,0.004070,I,0,/Users/fahim/Documents/GitHub/EKG Processing/d...,298
1,0,0.086,0.040,0.068,0.056,0.126,3.603565,-1.516245,0.012474,0.004790,I,1,/Users/fahim/Documents/GitHub/EKG Processing/d...,369
2,0,0.086,0.034,0.062,0.024,0.120,3.960839,-1.748594,0.008139,0.003427,I,2,/Users/fahim/Documents/GitHub/EKG Processing/d...,283
3,0,0.084,0.044,0.220,0.132,0.128,3.026500,-0.616297,0.013580,0.006610,I,3,/Users/fahim/Documents/GitHub/EKG Processing/d...,313
4,0,0.086,0.032,0.058,0.066,0.118,4.248869,-1.638104,0.008843,0.003420,I,4,/Users/fahim/Documents/GitHub/EKG Processing/d...,278
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2832,1,0.134,0.022,0.164,0.000,0.156,29.497283,-3.891611,0.077316,0.026943,V6,73,/Users/fahim/Documents/GitHub/EKG Processing/d...,352
2833,1,0.140,0.020,0.164,0.000,0.160,38.408115,-5.214948,0.096111,0.041877,V6,74,/Users/fahim/Documents/GitHub/EKG Processing/d...,394
2834,1,0.134,0.020,0.164,0.000,0.154,33.043695,-4.686000,0.069227,0.032536,V6,75,/Users/fahim/Documents/GitHub/EKG Processing/d...,470
2835,1,0.136,0.022,0.164,0.000,0.158,31.339620,-3.928361,0.074429,0.025640,V6,76,/Users/fahim/Documents/GitHub/EKG Processing/d...,342


In [53]:
processor.lead_I

array([ 1.27439024,  1.2804878 ,  1.2804878 , ..., -0.06402439,
       -0.06707317, -0.06707317])